## 1. Config

In [1]:
# =========================
# CONFIG
# =========================
# The notebook which trained the model is located in notebooks/LOG_REG_TF_IDF.ipynb
# It was trained with following config:

CONFIG = {
    "size": 100000,

    "word_max_features": 15000,
    "char_max_features": 15000,

    "min_df": 2,

    "word_ngram_range": (1, 2),
    "char_ngram_range": (2, 6),

    "lr": 0.3,
    "epochs": 40,
}

print("CONFIGURATION:")
for k, v in CONFIG.items():
    print(f"{k}: {v}")

CONFIGURATION:
size: 100000
word_max_features: 15000
char_max_features: 15000
min_df: 2
word_ngram_range: (1, 2)
char_ngram_range: (2, 6)
lr: 0.3
epochs: 40


## 2. Imports

In [2]:
import numpy as np
import pandas as pd

from src.dataloader.dataloader import SentenceDataModule
from src.tfidf.tfidf import TfIdfVectorizerNumpy
from src.models.logreg import SoftmaxLogReg
from src.train.train_logreg import save_model

## 3. Load dataset

from pathlib import Path

BASE_DIR = Path().resolve().parent  # move out of notebooks/
DATA_PATH = BASE_DIR / "datasets" / "records_long.json"

print("\nLoading dataset...")

dm = SentenceDataModule(
    record_path=str(DATA_PATH),
    size=CONFIG["size"],
    split=(70, 20, 10),
)

train_samples = dm.get_train_loader().samples
val_samples = dm.get_val_loader().samples
test_samples = dm.get_test_loader().samples

print("Train size:", len(train_samples))
print("Val size:", len(val_samples))
print("Test size:", len(test_samples))

## 4. Extract text + labels

X_train_text = [s["text"] for s in train_samples]
y_train = np.array([s["model"] for s in train_samples])

label_to_id = {
    "human": 0,
    "chatgpt": 1,
    "mistral": 2,
    "gemma": 3,
    "llama": 4
}

id_to_label = {v: k for k, v in label_to_id.items()}

total = len(y_train)

unique, counts = np.unique(y_train, return_counts=True)

print("\nTraining distribution:")
for u, c in zip(unique, counts):
    pct = (c / total) * 100
    print(f"{id_to_label[u]:10s}: {c:6d} ({pct:5.2f}%)")

X_val_text = [s["text"] for s in val_samples]
y_val = np.array([s["model"] for s in val_samples])

X_test_text = [s["text"] for s in test_samples]
y_test = np.array([s["model"] for s in test_samples])

print("\nExample sample:")
print(X_train_text[0])
print("Label:", y_train[0])

## 5.1 WORD TF-IDF

print("\nBuilding WORD TF-IDF...")

word_vectorizer = TfIdfVectorizerNumpy(
    max_features=CONFIG["word_max_features"],
    min_df=CONFIG["min_df"],
    ngram_range=CONFIG["word_ngram_range"],
    analyzer="word",
)

X_train_word = word_vectorizer.fit_transform(X_train_text)
X_val_word = word_vectorizer.transform(X_val_text)
X_test_word = word_vectorizer.transform(X_test_text)

print("Word TF-IDF shape:", X_train_word.shape)

## 5.2 Char TF-IDF

print("\nBuilding CHAR TF-IDF...")

char_vectorizer = TfIdfVectorizerNumpy(
    max_features=CONFIG["char_max_features"],
    min_df=CONFIG["min_df"],
    ngram_range=CONFIG["char_ngram_range"],
    analyzer="char",
)

X_train_char = char_vectorizer.fit_transform(X_train_text)
X_val_char = char_vectorizer.transform(X_val_text)
X_test_char = char_vectorizer.transform(X_test_text)

print("Char TF-IDF shape:", X_train_char.shape)

## 7. Combine features and add length

print("\nCombining features...")

X_train = np.hstack([X_train_word, X_train_char])
X_val = np.hstack([X_val_word, X_val_char])
X_test = np.hstack([X_test_word, X_test_char])

print("Combined shape:", X_train.shape)

print("\nAdding length feature...")

length_train = np.array([len(t) for t in X_train_text]).reshape(-1, 1) / 200
length_val = np.array([len(t) for t in X_val_text]).reshape(-1, 1) / 200
length_test = np.array([len(t) for t in X_test_text]).reshape(-1, 1) / 200

X_train = np.hstack([X_train, length_train])
X_val = np.hstack([X_val, length_val])
X_test = np.hstack([X_test, length_test])

print("Final feature shape:", X_train.shape)

## 8. Train model and save it

print("\nTraining model...")

clf = SoftmaxLogReg(
    input_dim=X_train.shape[1],
    num_classes=5,
    lr=CONFIG["lr"],
)

clf.fit(
    X_train,
    y_train,
    X_val=X_val,
    y_val=y_val,
    epochs=CONFIG["epochs"],
)

MODEL_PATH = Path("../models/subm1-g5-MEI-A.pkl")

save_model(
    model=clf,
    word_vectorizer=word_vectorizer,
    char_vectorizer=char_vectorizer,
    path=MODEL_PATH,
    config=CONFIG,
    label_to_id=label_to_id,
    save_full_model=True
)

## 9. Evaluate

print("\nEvaluating on test set...")

preds = clf.predict(X_test)
acc = np.mean(preds == y_test)

print("Test accuracy:", acc)

## 10. Test the given subm1.csv

In [ ]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

from src.tfidf.tfidf import TfIdfVectorizerNumpy

print("\nLoad model...")

def load_model(path):
    with open(path, "rb") as f:
        return pickle.load(f)

BASE_DIR = Path("..")
MODEL_PATH = Path("../models/subm1-g5-MEI-A.pkl")
model_data = load_model(MODEL_PATH)


print("\nRebuilding vectorizers...")

word_vectorizer = TfIdfVectorizerNumpy()
word_vectorizer.vocab = model_data["word_vocab"]
word_vectorizer.idf = model_data["word_idf"]

char_vectorizer = TfIdfVectorizerNumpy(analyzer="char")
char_vectorizer.vocab = model_data["char_vocab"]
char_vectorizer.idf = model_data["char_idf"]


clf = model_data["model"]



print("\nLoading CSV...")

df = pd.read_csv(
    str(BASE_DIR) + "/tests/TestData/subm1.csv",
    sep=";"
)

texts = df["Text"].tolist()

print("Number of samples:", len(texts))


print("\nBuilding features for CSV...")

X_word = word_vectorizer.transform(texts)
X_char = char_vectorizer.transform(texts)

X = np.hstack([X_word, X_char])

length = np.array([len(t) for t in texts]).reshape(-1, 1) / 200
X = np.hstack([X, length])

print("CSV feature shape:", X.shape)


id_to_company = {
    0: "Human",
    1: "OpenAI",
    2: "Anthropic",
    3: "Google",
    4: "Meta"
}


print("\nRunning predictions...")

preds = clf.predict(X)

df["Label"] = [id_to_company[p] for p in preds]

print(df.head())



df[["ID", "Label"]].to_csv("subm1-g5-MEI-A.csv", index=False)
print("Saved: subm1-g5-MEI-A.csv")


# import pickle
# from pathlib import Path
#
# print("\nLoad model...")
#
# def load_model(path):
#     with open(path, "rb") as f:
#         return pickle.load(f)
#
# MODEL_PATH = Path("../models/subm1-g5-MEI-A.pkl")
# model_data = load_model(MODEL_PATH)
#
#
# clf = model_data["model"]
# preds = clf.predict(X)
#
#
#
# print("\nLoading CSV...")
#
# df = pd.read_csv(
#     str(BASE_DIR) + "/tests/TestData/subm1.csv",
#     sep=";",
# )
#
# texts = df["Text"].tolist()
#
# print("Number of samples:", len(texts))
#
#
#
#
# print("\nBuilding features for CSV...")
#
# X_word = word_vectorizer.transform(texts)
# X_char = char_vectorizer.transform(texts)
#
# X = np.hstack([X_word, X_char])
#
# length = np.array([len(t) for t in texts]).reshape(-1, 1) / 200
# X = np.hstack([X, length])
#
# print("CSV feature shape:", X.shape)
#
# label_to_id = {
#     "human": 0,
#     "chatgpt": 1,
#     "mistral": 2,
#     "gemma": 3,
#     "llama": 4
# }
#
# id_to_company = {
#     0: "Human",
#     1: "OpenAI",
#     2: "Anthropic",
#     3: "Google",
#     4: "Meta"
# }
#
#
#
#
# print("\nRunning predictions...")
#
# preds = clf.predict(X)
#
# df["Label"] = [id_to_company[p] for p in preds]
#
# print(df.head())
#
#
#
#
# df.to_csv("subm1-g5-MEI-A.csv", index=False)
# print("subm1-g5-MEI-A.csv")